# Human-in-the-Loop トポニム解析ワークフロー

このノートブックは、非エンジニアでも扱える**アマゾン地名解析・遺跡候補抽出システム**のメインインターフェースです。

## 🎯 このノートブックでできること

1. **ハーモナイザー実行**: アマゾン地名の自動収集・正規化・候補地抽出
2. **結果可視化**: 地図上で候補地を確認
3. **Inspector実行**: AI による診断・改善提案
4. **改善計画作成**: GUI での設定変更・次回実行用YAML作成

## 📋 使用手順

1. **セクション1**: パラメータ設定
2. **セクション2**: ハーモナイザー実行
3. **セクション3**: 結果確認（地図表示）
4. **セクション4**: Inspector診断実行
5. **セクション5**: 改善計画作成

---

## 1. 📦 環境セットアップ

In [1]:
# 必要なライブラリのインポート
import os
import sys
import subprocess
import pandas as pd
import geopandas as gpd
import folium
import yaml
import json
import datetime
from pathlib import Path
from IPython.display import display, HTML, Markdown, clear_output
import ipywidgets as widgets
from ipywidgets import Layout, VBox, HBox, Button, Text, Textarea, Dropdown, FloatSlider, IntSlider, HTML as HTMLWidget

# プロジェクトルートの設定
PROJECT_ROOT = Path.cwd().parent
sys.path.append(str(PROJECT_ROOT / "src"))

print("✅ 環境セットアップ完了")
print(f"📁 プロジェクトルート: {PROJECT_ROOT}")
print(f"🐍 Python バージョン: {sys.version}")

# ディレクトリ構造の確認
important_dirs = ['data/raw', 'data/interim', 'data/output', 'scripts', 'actions']
for dir_path in important_dirs:
    full_path = PROJECT_ROOT / dir_path
    if full_path.exists():
        print(f"✅ {dir_path} 存在")
    else:
        full_path.mkdir(parents=True, exist_ok=True)
        print(f"📁 {dir_path} 作成")

✅ 環境セットアップ完了
📁 プロジェクトルート: /Users/kisamorikeiichi/Development/tamagawa_to_z
🐍 Python バージョン: 3.10.17 (main, Apr  8 2025, 12:10:59) [Clang 15.0.0 (clang-1500.1.0.2.5)]
✅ data/raw 存在
✅ data/interim 存在
✅ data/output 存在
✅ scripts 存在
✅ actions 存在


## 2. ⚙️ パラメータ設定

In [2]:
# パラメータ設定用のGUIウィジェット
print("⚙️ 解析パラメータを設定してください")

# 地域設定
bbox_widget = Text(
    value="-70.0 -11.0 -68.0 -9.0",
    description="🗺️ 解析地域 (BBox):",
    placeholder="lon_min lat_min lon_max lat_max",
    style={'description_width': '150px'},
    layout=Layout(width='400px')
)

# 解析パラメータ
dist_threshold_widget = FloatSlider(
    value=3.0,
    min=1.0,
    max=10.0,
    step=0.5,
    description="🌊 河川距離閾値 (km):",
    style={'description_width': '150px'},
    layout=Layout(width='400px')
)

occ_threshold_widget = IntSlider(
    value=5,
    min=1,
    max=20,
    step=1,
    description="💧 水域頻度閾値 (%):",
    style={'description_width': '150px'},
    layout=Layout(width='400px')
)

# LLMサンプルサイズ（コスト制御）
llm_sample_widget = IntSlider(
    value=100,
    min=10,
    max=500,
    step=10,
    description="🤖 LLMサンプルサイズ:",
    style={'description_width': '150px'},
    layout=Layout(width='400px')
)

# 可視化オプション
visualize_widget = widgets.Checkbox(
    value=False,
    description="📊 詳細可視化を有効にする",
    style={'description_width': '200px'}
)

# 水域頻度計算スキップ（高速テスト用）
skip_water_widget = widgets.Checkbox(
    value=False,
    description="⚡ 水域頻度計算をスキップ（高速テスト）",
    style={'description_width': '200px'}
)

# パラメータ表示
param_box = VBox([
    HTMLWidget(value="<h3>📍 解析地域設定</h3>"),
    bbox_widget,
    HTMLWidget(value="<h3>🔧 解析パラメータ</h3>"),
    dist_threshold_widget,
    occ_threshold_widget,
    llm_sample_widget,
    HTMLWidget(value="<h3>⚡ 実行オプション</h3>"),
    visualize_widget,
    skip_water_widget
])

display(param_box)

# パラメータ確認用の関数
def get_current_params():
    bbox_coords = [float(x) for x in bbox_widget.value.split()]
    return {
        'bbox': bbox_coords,
        'dist_threshold_km': dist_threshold_widget.value,
        'occ_pct_threshold': occ_threshold_widget.value,
        'llm_sample_size': llm_sample_widget.value,
        'visualize': visualize_widget.value,
        'skip_water_freq': skip_water_widget.value
    }

⚙️ 解析パラメータを設定してください


## 3. 🔄 ハーモナイザー実行

上記で設定したパラメータで `run_harmonizer.py` を実行し、地名収集・正規化・候補地抽出を行います。

In [3]:
# ハーモナイザー実行ボタン
run_harmonizer_btn = Button(
    description="🚀 ハーモナイザー実行",
    button_style='primary',
    layout=Layout(width='200px', height='40px')
)

# キャンセルボタン
cancel_harmonizer_btn = Button(
    description="⛔ キャンセル",
    button_style='danger',
    layout=Layout(width='100px', height='40px'),
    disabled=True
)

# 実行状況表示用
harmonizer_output = widgets.Output()
harmonizer_status = HTMLWidget(value="")

# グローバル変数でプロセス管理
current_harmonizer_process = None

def run_harmonizer(button):
    """ハーモナイザーを実行する関数"""
    global current_harmonizer_process
    
    with harmonizer_output:
        clear_output(wait=True)
        
        # パラメータ取得
        params = get_current_params()
        
        # パラメータ検証
        validation_errors = []
        try:
            if len(params['bbox']) != 4:
                validation_errors.append("BBox座標は4つの数値が必要です")
            
            # 座標範囲の妥当性チェック
            bbox = params['bbox']
            if not (-180 <= bbox[0] <= 180 and -180 <= bbox[2] <= 180):
                validation_errors.append("経度は-180から180の範囲で指定してください")
            if not (-90 <= bbox[1] <= 90 and -90 <= bbox[3] <= 90):
                validation_errors.append("緯度は-90から90の範囲で指定してください")
            if bbox[0] >= bbox[2] or bbox[1] >= bbox[3]:
                validation_errors.append("BBoxの座標順序が正しくありません (lon_min < lon_max, lat_min < lat_max)")
                
        except (ValueError, IndexError):
            validation_errors.append("BBox座標の形式が正しくありません")
        
        if params['dist_threshold_km'] <= 0:
            validation_errors.append("河川距離閾値は正の値である必要があります")
        if params['llm_sample_size'] <= 0:
            validation_errors.append("LLMサンプルサイズは正の値である必要があります")
        if params['occ_pct_threshold'] < 0 or params['occ_pct_threshold'] > 100:
            validation_errors.append("水域頻度閾値は0-100の範囲で指定してください")

        if validation_errors:
            print("❌ パラメータエラー:")
            for error in validation_errors:
                print(f"  • {error}")
            harmonizer_status.value = "<div style='color: red;'>❌ パラメータエラー</div>"
            return
        
        # UI状態更新
        run_harmonizer_btn.disabled = True
        cancel_harmonizer_btn.disabled = False
        harmonizer_status.value = "<div style='color: orange;'>🔄 実行中...</div>"
        
        print("🚀 ハーモナイザーを開始します")
        print("=" * 50)
        print(f"📍 解析領域: {params['bbox']}")
        print(f"🌊 河川距離閾値: {params['dist_threshold_km']} km")
        print(f"💧 水域頻度閾値: {params['occ_pct_threshold']} %")
        print(f"🤖 LLMサンプルサイズ: {params['llm_sample_size']}")
        print("=" * 50)
        
        try:
            # コマンド構築
            cmd = [
                "python", str(PROJECT_ROOT / "scripts/run_harmonizer.py"),
                "--bbox", *[str(x) for x in params['bbox']],
                "--llm-sample-size", str(params['llm_sample_size']),
                "--output_path", str(PROJECT_ROOT / "data/interim/acre_candidates.csv")
            ]
            
            if params['visualize']:
                cmd.append("--visualize")
            
            if params['skip_water_freq']:
                cmd.append("--skip-water-freq")
            
            print(f"🔧 実行コマンド: {' '.join(cmd)}")
            
            # 実行開始時刻を記録
            start_time = datetime.datetime.now()
            print(f"⏰ 開始時刻: {start_time.strftime('%Y-%m-%d %H:%M:%S')}")
            print()
            
            # 実行
            import subprocess
            current_harmonizer_process = subprocess.Popen(
                cmd,
                cwd=PROJECT_ROOT,
                stdout=subprocess.PIPE,
                stderr=subprocess.PIPE,
                text=True,
                bufsize=1,
                universal_newlines=True
            )
            
            # プロセス完了まで待機（タイムアウト付き）
            try:
                stdout, stderr = current_harmonizer_process.communicate(timeout=1800)  # 30分
                returncode = current_harmonizer_process.returncode
            except subprocess.TimeoutExpired:
                current_harmonizer_process.kill()
                stdout, stderr = current_harmonizer_process.communicate()
                print("⏰ 処理がタイムアウトしました（30分以上）")
                harmonizer_status.value = "<div style='color: red;'>⏰ タイムアウト</div>"
                return
            finally:
                # UI状態をリセット
                run_harmonizer_btn.disabled = False
                cancel_harmonizer_btn.disabled = True
                current_harmonizer_process = None
            
            # 実行時間を計算
            end_time = datetime.datetime.now()
            execution_time = end_time - start_time
            print(f"⏱️ 実行時間: {execution_time}")
            print(f"🏁 終了時刻: {end_time.strftime('%Y-%m-%d %H:%M:%S')}")
            
            # 結果表示
            if returncode == 0:
                print("\\n✅ ハーモナイザー実行完了!")
                harmonizer_status.value = "<div style='color: green;'>✅ 実行完了</div>"
                
                # 標準出力
                if stdout:
                    print("\\n📄 実行ログ:")
                    # 長いログの場合は最後の部分のみ表示
                    log_lines = stdout.strip().split('\\n')
                    if len(log_lines) > 20:
                        print("... (ログが長いため、最後の20行のみ表示)")
                        log_lines = log_lines[-20:]
                    print('\\n'.join(log_lines))
                
                # 出力ファイル確認
                output_file = PROJECT_ROOT / "data/interim/acre_candidates.csv"
                if output_file.exists():
                    print(f"\\n📁 出力ファイル: {output_file}")
                    
                    # ファイルサイズ表示
                    file_size = output_file.stat().st_size
                    print(f"📏 ファイルサイズ: {file_size:,} bytes ({file_size/1024:.1f} KB)")
                    
                    # データの詳細確認
                    try:
                        df = pd.read_csv(output_file)
                        print(f"📊 候補地点数: {len(df)} 件")
                        if len(df) > 0:
                            print(f"📋 データカラム: {', '.join(df.columns)}")
                            
                            # データ品質チェック
                            if 'total_score' in df.columns:
                                high_score_count = len(df[df['total_score'] > 0.7])
                                medium_score_count = len(df[(df['total_score'] >= 0.4) & (df['total_score'] <= 0.7)])
                                low_score_count = len(df[df['total_score'] < 0.4])
                                print(f"🎯 スコア別候補数:")
                                print(f"   高スコア(>0.7): {high_score_count} 件")
                                print(f"   中スコア(0.4-0.7): {medium_score_count} 件")
                                print(f"   低スコア(<0.4): {low_score_count} 件")
                            
                            if 'type' in df.columns:
                                type_summary = df['type'].value_counts().head(3)
                                print(f"📈 主要タイプ: {dict(type_summary)}")
                            
                            if 'dist_km' in df.columns:
                                avg_dist = df['dist_km'].mean()
                                print(f"🌊 平均河川距離: {avg_dist:.2f} km")
                            
                            # サンプルデータ表示
                            if len(df) <= 5:
                                print("\\n📋 全候補地点:")
                                display(df[['name', 'type', 'dist_km', 'total_score']].round(3) if 'total_score' in df.columns else df)
                            else:
                                print("\\n📋 上位5候補地点:")
                                if 'total_score' in df.columns:
                                    top_candidates = df.nlargest(5, 'total_score')[['name', 'type', 'dist_km', 'total_score']].round(3)
                                else:
                                    top_candidates = df.head(5)
                                display(top_candidates)
                                
                    except Exception as e:
                        print(f"⚠️ データ確認エラー: {e}")
                else:
                    print("⚠️ 出力ファイルが見つかりません")
                    
            else:
                print(f"\\n❌ エラーが発生しました (終了コード: {returncode})")
                harmonizer_status.value = "<div style='color: red;'>❌ エラー発生</div>"
                
                # 一般的なエラーの診断
                if returncode == 1:
                    print("💡 一般的な実行エラー。環境設定や入力データを確認してください。")
                elif returncode == 2:
                    print("💡 コマンドライン引数エラー。パラメータ設定を確認してください。")
                elif returncode == 130:
                    print("💡 処理がキャンセルされました。")
                
                if stderr:
                    print("\\n🚨 エラーメッセージ:")
                    print(stderr)
                    
                    # よくあるエラーのヒント
                    if "FileNotFoundError" in stderr:
                        print("\\n💡 ヒント: ファイルが見つかりません。データファイルの配置を確認してください。")
                        print("   必要ファイル:")
                        print("   - data/raw/osm/norte-latest.osm.pbf")
                        print("   - data/raw/hydrorivers_sahydrorivers_sa/HydroRIVERS_v10_sa.shp")
                        print("   - data/raw/GSW_occurrence/occurrence_70W_10Sv1_4_2021.tif")
                    elif "MemoryError" in stderr:
                        print("\\n💡 ヒント: メモリ不足です。LLMサンプルサイズを小さくしてみてください。")
                    elif "API" in stderr and ("key" in stderr or "401" in stderr):
                        print("\\n💡 ヒント: OpenAI APIキーの設定を確認してください。")
                        print("   .envファイルにOPENAI_API_KEY=your_key_hereを追加")
                    elif "ModuleNotFoundError" in stderr:
                        print("\\n💡 ヒント: 必要なライブラリがインストールされていません。")
                        print("   pip install -r requirements.txt を実行してください。")
                
                if stdout:
                    print("\\n📄 標準出力:")
                    print(stdout)
        
        except KeyboardInterrupt:
            print("\\n⚠️ 処理がユーザーによって中断されました")
            harmonizer_status.value = "<div style='color: orange;'>⚠️ 中断済み</div>"
        
        except Exception as e:
            print(f"\\n❌ 予期しないエラーが発生しました: {e}")
            harmonizer_status.value = "<div style='color: red;'>❌ 予期しないエラー</div>"
            import traceback
            traceback.print_exc()
        
        finally:
            # 確実にUI状態をリセット
            run_harmonizer_btn.disabled = False
            cancel_harmonizer_btn.disabled = True
            current_harmonizer_process = None

def cancel_harmonizer(button):
    """実行中のハーモナイザーをキャンセルする関数"""
    global current_harmonizer_process
    
    if current_harmonizer_process and current_harmonizer_process.poll() is None:
        print("⚠️ 実行をキャンセル中...")
        current_harmonizer_process.terminate()
        
        # プロセス終了を待機（最大5秒）
        try:
            current_harmonizer_process.wait(timeout=5)
        except subprocess.TimeoutExpired:
            # 強制終了
            current_harmonizer_process.kill()
            current_harmonizer_process.wait()
        
        print("✅ 実行をキャンセルしました")
        harmonizer_status.value = "<div style='color: orange;'>⚠️ キャンセル済み</div>"
        
        # UI状態をリセット
        run_harmonizer_btn.disabled = False
        cancel_harmonizer_btn.disabled = True
        current_harmonizer_process = None
    else:
        print("⚠️ キャンセルする実行中のプロセスがありません")

# ボタンにイベント設定
run_harmonizer_btn.on_click(run_harmonizer)
cancel_harmonizer_btn.on_click(cancel_harmonizer)

# 表示
display(VBox([
    HBox([run_harmonizer_btn, cancel_harmonizer_btn, harmonizer_status]),
    harmonizer_output
]))

## 4. 🗺️ 結果可視化（候補地マップ）

ハーモナイザーの実行結果を地図上で確認します。

In [ ]:
# 可視化実行ボタン
visualize_btn = Button(
    description="🗺️ 地図表示",
    button_style='info',
    layout=Layout(width='150px', height='40px')
)

# 地図表示用
map_output = widgets.Output()
map_status = HTMLWidget(value="")

def create_candidates_map(button):
    """候補地点の地図を作成する関数"""
    with map_output:
        clear_output(wait=True)
        
        map_status.value = "<div style='color: orange;'>🔄 地図作成中...</div>"
        
        try:
            # データファイルの確認
            candidates_file = PROJECT_ROOT / "data/interim/acre_candidates.csv"
            
            if not candidates_file.exists():
                print("❌ 候補データファイルが見つかりません。")
                print("💡 まず 'ハーモナイザー実行' を実行してください。")
                map_status.value = "<div style='color: red;'>❌ データなし</div>"
                return
            
            # CSVファイルの読み込み
            print("📖 候補データを読み込み中...")
            df = pd.read_csv(candidates_file)
            
            if len(df) == 0:
                print("⚠️ 候補データが空です")
                map_status.value = "<div style='color: orange;'>⚠️ データ空</div>"
                return
            
            print(f"📊 候補地点数: {len(df)} 件")
            
            # ジオメトリカラムの確認と変換
            if 'geometry' in df.columns:
                # WKT文字列からGeometryに変換
                gdf = gpd.GeoDataFrame(
                    df, 
                    geometry=gpd.GeoSeries.from_wkt(df['geometry']), 
                    crs="EPSG:4326"
                )
            elif 'longitude' in df.columns and 'latitude' in df.columns:
                # 経度緯度から Point 作成
                from shapely.geometry import Point
                gdf = gpd.GeoDataFrame(
                    df,
                    geometry=[Point(row.longitude, row.latitude) for _, row in df.iterrows()],
                    crs="EPSG:4326"
                )
            else:
                print("❌ 地理情報カラム（geometry または longitude/latitude）が見つかりません")
                map_status.value = "<div style='color: red;'>❌ 地理情報なし</div>"
                return
            
            # 地図の中心座標を計算
            center_lat = gdf.geometry.centroid.y.mean()
            center_lon = gdf.geometry.centroid.x.mean()
            
            print(f"🎯 地図中心: ({center_lat:.3f}, {center_lon:.3f})")
            
            # Folium地図作成
            m = folium.Map(
                location=[center_lat, center_lon],
                zoom_start=10,
                tiles='OpenStreetMap'
            )
            
            # スコア情報の確認
            has_score = 'total_score' in gdf.columns
            
            # 地点をマップに追加
            for idx, row in gdf.iterrows():
                # ポップアップ情報の構築
                popup_html = f"""
                <div style="width: 250px;">
                    <h4>{row.get('name', 'Unknown')}</h4>
                    <p><strong>正規化名:</strong> {row.get('normalized_name', 'N/A')}</p>
                    <p><strong>タイプ:</strong> {row.get('type', 'N/A')}</p>
                """
                
                if 'dist_km' in row:
                    popup_html += f"<p><strong>河川距離:</strong> {row['dist_km']:.1f} km</p>"
                
                if 'occ_pct' in row:
                    popup_html += f"<p><strong>水域頻度:</strong> {row['occ_pct']:.1f} %</p>"
                
                if has_score:
                    popup_html += f"<p><strong>総スコア:</strong> {row['total_score']:.2f}</p>"
                
                popup_html += "</div>"
                
                # マーカーサイズと色の決定
                if has_score:
                    score = row['total_score']
                    radius = 5 + score * 10  # スコアに応じてサイズ調整
                    color = 'red' if score > 0.7 else 'orange' if score > 0.4 else 'yellow'
                else:
                    radius = 8
                    color = 'blue'
                
                # CircleMarkerを追加
                folium.CircleMarker(
                    location=[row.geometry.y, row.geometry.x],
                    popup=folium.Popup(popup_html, max_width=300),
                    radius=radius,
                    color=color,
                    fill=True,
                    fillOpacity=0.7,
                    weight=2
                ).add_to(m)
            
            # 凡例の追加
            if has_score:
                legend_html = '''
                <div style="position: fixed; 
                            bottom: 50px; right: 50px; width: 150px; height: 90px; 
                            background-color: white; border:2px solid grey; z-index:9999; 
                            font-size:14px; padding: 10px">
                <h4>候補地スコア</h4>
                <p><i class="fa fa-circle" style="color:red"></i> 高 (>0.7)</p>
                <p><i class="fa fa-circle" style="color:orange"></i> 中 (0.4-0.7)</p>
                <p><i class="fa fa-circle" style="color:yellow"></i> 低 (<0.4)</p>
                </div>
                '''
                m.get_root().html.add_child(folium.Element(legend_html))
            
            # 統計情報の表示
            print("\n📊 候補地点統計:")
            print(f"  総数: {len(gdf)} 件")
            
            if 'type' in gdf.columns:
                type_counts = gdf['type'].value_counts()
                print("  タイプ別件数:")
                for type_name, count in type_counts.items():
                    print(f"    {type_name}: {count} 件")
            
            if has_score:
                score_stats = gdf['total_score'].describe()
                print(f"  スコア統計:")
                print(f"    平均: {score_stats['mean']:.3f}")
                print(f"    最大: {score_stats['max']:.3f}")
                print(f"    最小: {score_stats['min']:.3f}")
            
            print("\n🗺️ 地図を表示します（地点をクリックして詳細を確認）")
            
            # 地図表示
            display(m)
            
            map_status.value = "<div style='color: green;'>✅ 地図表示完了</div>"
            
        except Exception as e:
            print(f"❌ 地図作成中にエラーが発生しました: {e}")
            map_status.value = "<div style='color: red;'>❌ エラー</div>"
            import traceback
            traceback.print_exc()

# ボタンにイベント設定
visualize_btn.on_click(create_candidates_map)

# 表示
display(VBox([
    HBox([visualize_btn, map_status]),
    map_output
]))

## 5. 🔍 Inspector実行（AI診断）

候補地点の品質を評価し、改善提案を行います。

In [7]:
# Inspector実行ボタン
run_inspector_btn = Button(
    description="🔍 AI診断実行",
    button_style='warning',
    layout=Layout(width='150px', height='40px')
)

# Inspector キャンセルボタン
cancel_inspector_btn = Button(
    description="⛔ キャンセル",
    button_style='danger',
    layout=Layout(width='100px', height='40px'),
    disabled=True
)

# 実行状況表示用
inspector_output = widgets.Output()
inspector_status = HTMLWidget(value="")

# グローバル変数でプロセス管理
current_inspector_process = None

def run_inspector(button):
    """Inspector-Validator Agentを実行する関数"""
    global current_inspector_process
    
    with inspector_output:
        clear_output(wait=True)
        
        # UI状態更新
        run_inspector_btn.disabled = True
        cancel_inspector_btn.disabled = False
        inspector_status.value = "<div style='color: orange;'>🔄 AI診断実行中...</div>"
        
        try:
            # 入力ファイルの確認
            candidates_file = PROJECT_ROOT / "data/interim/acre_candidates.csv"
            known_sites_file = PROJECT_ROOT / "data/raw/known_sites.gpkg"
            
            if not candidates_file.exists():
                print("❌ 候補データファイルが見つかりません。")
                print("💡 まず 'ハーモナイザー実行' を実行してください。")
                inspector_status.value = "<div style='color: red;'>❌ 候補データなし</div>"
                return
            
            # 候補データの簡易検証
            try:
                df_candidates = pd.read_csv(candidates_file)
                if len(df_candidates) == 0:
                    print("❌ 候補データファイルが空です。")
                    print("💡 まず 'ハーモナイザー実行' を実行してください。")
                    inspector_status.value = "<div style='color: red;'>❌ 候補データ空</div>"
                    return
                print(f"📊 候補データ: {len(df_candidates)} 件を確認")
            except Exception as e:
                print(f"❌ 候補データファイルの読み込みエラー: {e}")
                inspector_status.value = "<div style='color: red;'>❌ 候補データエラー</div>"
                return
            
            if not known_sites_file.exists():
                print("⚠️ 既知遺跡データファイルが見つかりません。")
                print(f"📁 期待されるパス: {known_sites_file}")
                print("💡 サンプルファイルを作成して続行します...")
                
                # サンプル既知遺跡データの作成
                import geopandas as gpd
                from shapely.geometry import Point
                
                # 候補地点の範囲内にサンプルデータを作成
                try:
                    bbox_bounds = df_candidates[['longitude', 'latitude']].describe() if 'longitude' in df_candidates.columns else None
                    if bbox_bounds is not None:
                        lon_center = bbox_bounds.loc['mean', 'longitude']
                        lat_center = bbox_bounds.loc['mean', 'latitude']
                    else:
                        # デフォルト座標（アクレ州）
                        lon_center, lat_center = -69.0, -10.0
                    
                    sample_sites = gpd.GeoDataFrame({
                        'site_id': ['site_001', 'site_002', 'site_003', 'site_004', 'site_005'],
                        'name': ['Sample Site 1', 'Sample Site 2', 'Sample Site 3', 'Sample Site 4', 'Sample Site 5'],
                        'geometry': [
                            Point(lon_center + 0.1, lat_center + 0.1),
                            Point(lon_center - 0.1, lat_center - 0.1),
                            Point(lon_center + 0.2, lat_center - 0.2),
                            Point(lon_center - 0.2, lat_center + 0.2),
                            Point(lon_center, lat_center)
                        ],
                        'confidence': ['high', 'medium', 'high', 'low', 'medium']
                    }, crs="EPSG:4326")
                    
                    known_sites_file.parent.mkdir(parents=True, exist_ok=True)
                    sample_sites.to_file(known_sites_file, driver="GPKG")
                    print(f"✅ サンプル既知遺跡データを作成しました: {known_sites_file}")
                    print(f"📍 サンプルサイト数: {len(sample_sites)} 件")
                    
                except Exception as e:
                    print(f"⚠️ サンプルデータ作成エラー: {e}")
                    print("デフォルトのサンプルデータを作成します...")
                    
                    sample_sites = gpd.GeoDataFrame({
                        'site_id': ['site_001', 'site_002', 'site_003'],
                        'name': ['Sample Site 1', 'Sample Site 2', 'Sample Site 3'],
                        'geometry': [Point(-69.5, -10.5), Point(-69.0, -10.0), Point(-68.5, -9.5)]
                    }, crs="EPSG:4326")
                    
                    known_sites_file.parent.mkdir(parents=True, exist_ok=True)
                    sample_sites.to_file(known_sites_file, driver="GPKG")
                    print(f"✅ デフォルトサンプルを作成: {known_sites_file}")
            
            print("🔍 Inspector-Validator Agent を開始します")
            print("=" * 50)
            print(f"📊 候補データ: {candidates_file}")
            print(f"🏛️ 既知遺跡データ: {known_sites_file}")
            print("=" * 50)
            
            # 実行開始時刻を記録
            start_time = datetime.datetime.now()
            print(f"⏰ 開始時刻: {start_time.strftime('%Y-%m-%d %H:%M:%S')}")
            
            # コマンド構築
            cmd = [
                "python", str(PROJECT_ROOT / "scripts/run_inspector.py"),
                "--candidates", str(candidates_file),
                "--known", str(known_sites_file),
                "--output", str(PROJECT_ROOT / "data/output/inspector_reports"),
                "--verbose"
            ]
            
            # メタ情報ファイルの確認
            meta_file = PROJECT_ROOT / "config/run_meta.yaml"
            if meta_file.exists():
                cmd.extend(["--meta", str(meta_file)])
                print(f"📄 メタ情報ファイル: {meta_file}")
            
            # 辞書ファイルの確認
            dict_file = PROJECT_ROOT / "data/dict/toponym_dict.csv"
            if dict_file.exists():
                cmd.extend(["--dict", str(dict_file)])
                print(f"📚 辞書ファイル: {dict_file}")
            
            print(f"🔧 実行コマンド: {' '.join(cmd)}")
            print()
            
            # 実行
            import subprocess
            current_inspector_process = subprocess.Popen(
                cmd,
                cwd=PROJECT_ROOT,
                stdout=subprocess.PIPE,
                stderr=subprocess.PIPE,
                text=True,
                bufsize=1,
                universal_newlines=True
            )
            
            # プロセス完了まで待機（タイムアウト付き）
            try:
                stdout, stderr = current_inspector_process.communicate(timeout=600)  # 10分
                returncode = current_inspector_process.returncode
            except subprocess.TimeoutExpired:
                current_inspector_process.kill()
                stdout, stderr = current_inspector_process.communicate()
                print("⏰ 処理がタイムアウトしました（10分以上）")
                inspector_status.value = "<div style='color: red;'>⏰ タイムアウト</div>"
                return
            finally:
                # UI状態をリセット
                run_inspector_btn.disabled = False
                cancel_inspector_btn.disabled = True
                current_inspector_process = None
            
            # 実行時間を計算
            end_time = datetime.datetime.now()
            execution_time = end_time - start_time
            print(f"⏱️ 実行時間: {execution_time}")
            print(f"🏁 終了時刻: {end_time.strftime('%Y-%m-%d %H:%M:%S')}")
            
            # 結果表示
            if returncode == 0:
                print("\\n✅ Inspector診断完了!")
                inspector_status.value = "<div style='color: green;'>✅ 診断完了</div>"
                
                # 標準出力の処理
                if stdout:
                    print("\\n📄 診断結果:")
                    # 長い出力の場合は要約
                    output_lines = stdout.strip().split('\\n')
                    if len(output_lines) > 30:
                        print("... (出力が長いため、重要な部分のみ表示)")
                        # 重要な情報を抽出
                        important_lines = []
                        for line in output_lines:
                            if any(keyword in line for keyword in ['✅', '❌', '📊', '🎯', '💡', 'Recall', 'mAP', 'Workload']):
                                important_lines.append(line)
                        print('\\n'.join(important_lines[-15:]))  # 最後の15行
                    else:
                        print(stdout)
                
                # 出力ファイルの確認と詳細表示
                output_dir = PROJECT_ROOT / "data/output/inspector_reports"
                if output_dir.exists():
                    # 最新のレポートを検索
                    latest_reports = sorted(output_dir.glob("*/report_*.md"), key=lambda x: x.stat().st_mtime, reverse=True)
                    latest_results = sorted(output_dir.glob("*/results_*.json"), key=lambda x: x.stat().st_mtime, reverse=True)
                    
                    if latest_reports:
                        latest_report = latest_reports[0]
                        print(f"\\n📄 最新レポート: {latest_report}")
                        
                        # 対応するYAMLファイルも確認
                        yaml_file = latest_report.parent / latest_report.name.replace("report_", "plan_").replace(".md", ".yaml")
                        if yaml_file.exists():
                            print(f"📄 改善計画: {yaml_file}")
                            
                            # グローバル変数として最新ファイルパスを保存
                            global latest_report_path, latest_plan_path
                            latest_report_path = latest_report
                            latest_plan_path = yaml_file
                    
                    # JSONファイルからメトリクスを抽出して表示
                    if latest_results:
                        try:
                            with open(latest_results[0], 'r', encoding='utf-8') as f:
                                result_data = json.load(f)
                            
                            metrics = result_data.get('metrics', {})
                            if metrics:
                                print("\\n📊 主要メトリクス:")
                                
                                # Recall@100
                                if 'recall@100' in metrics:
                                    recall = metrics['recall@100']
                                    status = "🟢 良好" if recall >= 0.5 else "🟡 改善余地" if recall >= 0.3 else "🔴 低リコール"
                                    print(f"  🎯 Recall@100: {recall:.3f} ({status})")
                                
                                # mAP
                                if 'map' in metrics:
                                    map_score = metrics['map']
                                    status = "🟢 良好" if map_score >= 0.4 else "🟡 改善余地" if map_score >= 0.2 else "🔴 低精度"
                                    print(f"  🎯 mAP: {map_score:.3f} ({status})")
                                
                                # Workload
                                if 'workload' in metrics:
                                    workload = metrics['workload']
                                    status = "🟢 適正" if workload < 500 else "🟡 中負荷" if workload < 1000 else "🔴 高負荷"
                                    print(f"  📊 候補数: {workload} ({status})")
                                
                                # Diversity
                                if 'diversity' in metrics:
                                    diversity = metrics['diversity']
                                    print(f"  🌈 多様性: {diversity:.3f}")
                            
                            # 改善提案の表示
                            proposal = result_data.get('proposal', {})
                            if proposal:
                                print(f"\\n💡 改善提案:")
                                print(f"  アクション: {proposal.get('action', 'N/A')}")
                                print(f"  理由: {proposal.get('rationale', 'N/A')}")
                                if proposal.get('priority'):
                                    priority_icon = "🔴" if proposal['priority'] == 'high' else "🟡" if proposal['priority'] == 'medium' else "🟢"
                                    print(f"  優先度: {priority_icon} {proposal['priority']}")
                                    
                        except Exception as e:
                            print(f"⚠️ メトリクス表示エラー: {e}")
                    
                    print("\\n🎯 次のステップ:")
                    print("  1. 📄レポート表示 ボタンで詳細レポートを確認")
                    print("  2. 改善計画作成 セクションで設定を調整")
                    print("  3. 新しい設定でハーモナイザーを再実行")
                    
            else:
                print(f"\\n❌ エラーが発生しました (終了コード: {returncode})")
                inspector_status.value = "<div style='color: red;'>❌ エラー発生</div>"
                
                # 一般的なエラーの診断
                if returncode == 1:
                    print("💡 一般的な実行エラー。入力データや環境設定を確認してください。")
                elif returncode == 2:
                    print("💡 コマンドライン引数エラー。ファイルパスを確認してください。")
                elif returncode == 130:
                    print("💡 処理がキャンセルされました。")
                
                if stderr:
                    print("\\n🚨 エラーメッセージ:")
                    print(stderr)
                    
                    # よくあるエラーのヒント
                    if "FileNotFoundError" in stderr:
                        print("\\n💡 ヒント: ファイルが見つかりません。")
                        print("   - 候補データファイルが存在することを確認")
                        print("   - ハーモナイザーが正常に完了していることを確認")
                    elif "pandas" in stderr and "read_csv" in stderr:
                        print("\\n💡 ヒント: CSVファイルの形式に問題があります。")
                        print("   - ハーモナイザーの出力を再確認してください")
                    elif "API" in stderr and ("key" in stderr or "401" in stderr):
                        print("\\n💡 ヒント: OpenAI APIキーの設定を確認してください。")
                        print("   - .envファイルにOPENAI_API_KEY=your_key_hereを追加")
                    elif "geopandas" in stderr:
                        print("\\n💡 ヒント: 地理データの処理エラーです。")
                        print("   - 既知遺跡データファイルの形式を確認してください")
                
                if stdout:
                    print("\\n📄 標準出力:")
                    print(stdout)
        
        except KeyboardInterrupt:
            print("\\n⚠️ 処理がユーザーによって中断されました")
            inspector_status.value = "<div style='color: orange;'>⚠️ 中断済み</div>"
        
        except Exception as e:
            print(f"\\n❌ 予期しないエラーが発生しました: {e}")
            inspector_status.value = "<div style='color: red;'>❌ 予期しないエラー</div>"
            import traceback
            traceback.print_exc()
        
        finally:
            # 確実にUI状態をリセット
            run_inspector_btn.disabled = False
            cancel_inspector_btn.disabled = True
            current_inspector_process = None

def cancel_inspector(button):
    """実行中のInspectorをキャンセルする関数"""
    global current_inspector_process
    
    if current_inspector_process and current_inspector_process.poll() is None:
        print("⚠️ AI診断をキャンセル中...")
        current_inspector_process.terminate()
        
        # プロセス終了を待機（最大5秒）
        try:
            current_inspector_process.wait(timeout=5)
        except subprocess.TimeoutExpired:
            # 強制終了
            current_inspector_process.kill()
            current_inspector_process.wait()
        
        print("✅ AI診断をキャンセルしました")
        inspector_status.value = "<div style='color: orange;'>⚠️ キャンセル済み</div>"
        
        # UI状態をリセット
        run_inspector_btn.disabled = False
        cancel_inspector_btn.disabled = True
        current_inspector_process = None
    else:
        print("⚠️ キャンセルするAI診断がありません")

# ボタンにイベント設定
run_inspector_btn.on_click(run_inspector)
cancel_inspector_btn.on_click(cancel_inspector)

# 表示
display(VBox([
    HBox([run_inspector_btn, cancel_inspector_btn, inspector_status]),
    inspector_output
]))

## 6. 📄 診断レポート表示

Inspectorが生成したMarkdownレポートを表示します。

In [ ]:
# レポート表示ボタン
show_report_btn = Button(
    description="📄 レポート表示",
    button_style='info',
    layout=Layout(width='150px', height='40px')
)

# レポート表示用
report_output = widgets.Output()
report_status = HTMLWidget(value="")

def show_latest_report(button):
    """最新の診断レポートを表示する関数"""
    with report_output:
        clear_output(wait=True)
        
        report_status.value = "<div style='color: orange;'>🔄 レポート読み込み中...</div>"
        
        try:
            # 最新レポートファイルの検索
            output_dir = PROJECT_ROOT / "data/output/inspector_reports"
            
            if not output_dir.exists():
                print("❌ レポートディレクトリが見つかりません。")
                print("💡 まず 'AI診断実行' を実行してください。")
                report_status.value = "<div style='color: red;'>❌ レポートなし</div>"
                return
            
            # レポートファイルを検索（最新順）
            report_files = sorted(output_dir.glob("*/report_*.md"), key=lambda x: x.stat().st_mtime, reverse=True)
            
            if not report_files:
                print("❌ レポートファイルが見つかりません。")
                print("💡 まず 'AI診断実行' を実行してください。")
                report_status.value = "<div style='color: red;'>❌ レポートファイルなし</div>"
                return
            
            latest_report = report_files[0]
            print(f"📄 最新レポート: {latest_report}")
            print(f"🕒 作成日時: {datetime.datetime.fromtimestamp(latest_report.stat().st_mtime)}")
            print("=" * 80)
            
            # Markdownファイルの読み込みと表示
            with open(latest_report, 'r', encoding='utf-8') as f:
                markdown_content = f.read()
            
            # Markdownとして表示
            display(Markdown(markdown_content))
            
            report_status.value = "<div style='color: green;'>✅ レポート表示完了</div>"
            
            # グローバル変数として保存（次のセルで使用）
            global latest_report_path
            latest_report_path = latest_report
            
        except Exception as e:
            print(f"❌ レポート読み込み中にエラーが発生しました: {e}")
            report_status.value = "<div style='color: red;'>❌ エラー</div>"
            import traceback
            traceback.print_exc()

# ボタンにイベント設定
show_report_btn.on_click(show_latest_report)

# 表示
display(VBox([
    HBox([show_report_btn, report_status]),
    report_output
]))

## 7. ⚙️ 改善アクション編集GUI

AI診断結果を基に、次回実行用の改善計画YAMLを作成・編集します。

In [10]:
# 改善アクション編集GUI
print("⚙️ 改善計画を作成してください")

# アクション選択
action_widget = Dropdown(
    options=[
        ('パラメータ調整', 'set_param'),
        ('除外マスク追加', 'add_exclude_mask'), 
        ('語根重み調整', 'add_root_weight')
    ],
    value='set_param',
    description='🎯 改善アクション:',
    style={'description_width': '120px'},
    layout=Layout(width='300px')
)

# パラメータ入力エリア
params_widget = Textarea(
    value="dist_threshold_km: 4.0\nocc_pct_threshold: 3",
    placeholder='YAML形式でパラメータを入力...',
    description='📝 パラメータ:',
    style={'description_width': '120px'},
    layout=Layout(width='500px', height='120px')
)

# 変更理由入力
rationale_widget = Textarea(
    value='候補地点数が多すぎるため、閾値を厳しくして精度を向上させる',
    placeholder='変更理由を入力...',
    description='💭 変更理由:',
    style={'description_width': '120px'},
    layout=Layout(width='500px', height='80px')
)

# プリセット設定ボタン
preset_high_precision = Button(
    description="🎯 高精度設定",
    button_style='success',
    layout=Layout(width='120px')
)

preset_high_recall = Button(
    description="🌐 高再現率設定",
    button_style='warning',
    layout=Layout(width='120px')
)

preset_balanced = Button(
    description="⚖️ バランス設定",
    button_style='info',
    layout=Layout(width='120px')
)

# プリセット設定関数
def set_high_precision(button):
    action_widget.value = 'set_param'
    params_widget.value = "dist_threshold_km: 2.0\nocc_pct_threshold: 8"
    rationale_widget.value = "高精度設定：厳しい閾値で偽陽性を減らし、確実性の高い候補のみを抽出"

def set_high_recall(button):
    action_widget.value = 'set_param'
    params_widget.value = "dist_threshold_km: 5.0\nocc_pct_threshold: 2"
    rationale_widget.value = "高再現率設定：緩い閾値でより多くの候補を発見し、見逃しを減らす"

def set_balanced(button):
    action_widget.value = 'set_param'
    params_widget.value = "dist_threshold_km: 3.0\nocc_pct_threshold: 5"
    rationale_widget.value = "バランス設定：精度と再現率のバランスを取った標準的な閾値"

preset_high_precision.on_click(set_high_precision)
preset_high_recall.on_click(set_high_recall)
preset_balanced.on_click(set_balanced)

# 計画保存ボタン
save_plan_btn = Button(
    description='💾 計画保存',
    button_style='primary',
    layout=Layout(width='120px', height='40px')
)

# 保存状況表示
save_status = HTMLWidget(value="")
save_output = widgets.Output()

def save_improvement_plan(button):
    """改善計画をYAMLファイルとして保存する関数"""
    with save_output:
        clear_output(wait=True)
        
        save_status.value = "<div style='color: orange;'>💾 保存中...</div>"
        
        try:
            # パラメータをYAMLとしてパース
            try:
                params_dict = yaml.safe_load(params_widget.value)
            except yaml.YAMLError as e:
                print(f"❌ パラメータのYAML形式が正しくありません: {e}")
                save_status.value = "<div style='color: red;'>❌ YAML形式エラー</div>"
                return
            
            # 改善計画の構築
            improvement_plan = {
                'action': action_widget.value,
                'params': params_dict,
                'rationale': rationale_widget.value,
                'created_at': datetime.datetime.now().isoformat(),
                'created_by': 'human-in-the-loop'
            }
            
            # ファイル名の生成
            timestamp = datetime.datetime.now().strftime("%Y-%m-%d_%H-%M-%S")
            filename = f"plan_{timestamp}.yaml"
            
            # actionsディレクトリに保存
            actions_dir = PROJECT_ROOT / "actions"
            actions_dir.mkdir(exist_ok=True)
            
            plan_file = actions_dir / filename
            
            # YAMLファイルとして保存
            with open(plan_file, 'w', encoding='utf-8') as f:
                yaml.safe_dump(improvement_plan, f, default_flow_style=False, allow_unicode=True)
            
            print(f"✅ 改善計画を保存しました: {plan_file}")
            print("\n📋 保存内容:")
            print(f"  🎯 アクション: {improvement_plan['action']}")
            print(f"  📝 パラメータ: {improvement_plan['params']}")
            print(f"  💭 理由: {improvement_plan['rationale']}")
            
            save_status.value = "<div style='color: green;'>✅ 保存完了</div>"
            
            print("\n🎯 次のステップ:")
            print("  1. 保存されたYAMLファイルをレビュー")
            print("  2. 必要に応じてパラメータを手動調整")
            print("  3. 新しい設定でハーモナイザーを再実行")
            
        except Exception as e:
            print(f"❌ 計画保存中にエラーが発生しました: {e}")
            save_status.value = "<div style='color: red;'>❌ 保存エラー</div>"
            import traceback
            traceback.print_exc()

save_plan_btn.on_click(save_improvement_plan)

# レイアウト構築
action_box = VBox([
    HTMLWidget(value="<h3>🎯 改善アクション設定</h3>"),
    action_widget,
    HTMLWidget(value="<h4>📝 パラメータ設定</h4>"),
    params_widget,
    HTMLWidget(value="<h4>🎭 プリセット設定</h4>"),
    HBox([preset_high_precision, preset_high_recall, preset_balanced]),
    HTMLWidget(value="<h4>💭 変更理由</h4>"),
    rationale_widget,
    HTMLWidget(value="<h4>💾 保存</h4>"),
    HBox([save_plan_btn, save_status]),
    save_output
])

display(action_box)

⚙️ 改善計画を作成してください


## 8. 📊 メトリクス可視化

過去の実行履歴からメトリクスのトレンドを表示します（開発中）。

In [11]:
# メトリクス表示ボタン
show_metrics_btn = Button(
    description="📊 メトリクス表示",
    button_style='info',
    layout=Layout(width='150px', height='40px')
)

# メトリクス表示用
metrics_output = widgets.Output()
metrics_status = HTMLWidget(value="")

def show_metrics_trends(button):
    """メトリクス履歴を表示する関数"""
    with metrics_output:
        clear_output(wait=True)
        
        metrics_status.value = "<div style='color: orange;'>📊 メトリクス集計中...</div>"
        
        try:
            # 過去のレポート結果を収集
            output_dir = PROJECT_ROOT / "data/output/inspector_reports"
            
            if not output_dir.exists():
                print("❌ レポートディレクトリが見つかりません。")
                metrics_status.value = "<div style='color: red;'>❌ データなし</div>"
                return
            
            # JSON結果ファイルを検索
            result_files = sorted(output_dir.glob("*/results_*.json"), key=lambda x: x.stat().st_mtime)
            
            if not result_files:
                print("📊 メトリクス履歴がまだありません。")
                print("💡 AI診断を複数回実行すると、ここにトレンドが表示されます。")
                metrics_status.value = "<div style='color: orange;'>📊 履歴なし</div>"
                return
            
            print(f"📊 {len(result_files)} 件の実行履歴を発見")
            
            # メトリクスデータの収集
            metrics_history = []
            
            for result_file in result_files:
                try:
                    with open(result_file, 'r', encoding='utf-8') as f:
                        result_data = json.load(f)
                    
                    # タイムスタンプとメトリクスを抽出
                    timestamp = result_data.get('timestamp', result_file.stat().st_mtime)
                    metrics = result_data.get('metrics', {})
                    
                    metrics_history.append({
                        'timestamp': timestamp,
                        'recall@100': metrics.get('recall@100', 0),
                        'map': metrics.get('map', 0),
                        'workload': metrics.get('workload', 0),
                        'diversity': metrics.get('diversity', 0)
                    })
                    
                except Exception as e:
                    print(f"⚠️ {result_file} の読み込みに失敗: {e}")
                    continue
            
            if not metrics_history:
                print("❌ 有効なメトリクスデータが見つかりませんでした。")
                metrics_status.value = "<div style='color: red;'>❌ データ無効</div>"
                return
            
            # DataFrameに変換
            df_metrics = pd.DataFrame(metrics_history)
            df_metrics['timestamp'] = pd.to_datetime(df_metrics['timestamp'])
            df_metrics = df_metrics.sort_values('timestamp')
            
            print("📈 メトリクス履歴表:")
            display(df_metrics)
            
            # 簡易的な可視化（matplotlib使用）
            if len(df_metrics) > 1:
                import matplotlib.pyplot as plt
                
                fig, axes = plt.subplots(2, 2, figsize=(12, 8))
                fig.suptitle('メトリクス履歴トレンド', fontsize=16)
                
                # Recall@100
                axes[0, 0].plot(df_metrics['timestamp'], df_metrics['recall@100'], 'o-', color='blue')
                axes[0, 0].set_title('Recall@100')
                axes[0, 0].set_ylabel('Recall')
                axes[0, 0].grid(True, alpha=0.3)
                
                # mAP
                axes[0, 1].plot(df_metrics['timestamp'], df_metrics['map'], 'o-', color='green')
                axes[0, 1].set_title('mAP (Mean Average Precision)')
                axes[0, 1].set_ylabel('mAP')
                axes[0, 1].grid(True, alpha=0.3)
                
                # Workload
                axes[1, 0].plot(df_metrics['timestamp'], df_metrics['workload'], 'o-', color='orange')
                axes[1, 0].set_title('候補数 (Workload)')
                axes[1, 0].set_ylabel('件数')
                axes[1, 0].grid(True, alpha=0.3)
                
                # Diversity
                axes[1, 1].plot(df_metrics['timestamp'], df_metrics['diversity'], 'o-', color='purple')
                axes[1, 1].set_title('多様性 (Diversity)')
                axes[1, 1].set_ylabel('Shannon Entropy')
                axes[1, 1].grid(True, alpha=0.3)
                
                plt.tight_layout()
                plt.show()
                
                # 最新の改善状況
                if len(df_metrics) >= 2:
                    latest = df_metrics.iloc[-1]
                    previous = df_metrics.iloc[-2]
                    
                    print("\n📈 前回からの変化:")
                    
                    recall_change = latest['recall@100'] - previous['recall@100']
                    map_change = latest['map'] - previous['map']
                    workload_change = latest['workload'] - previous['workload']
                    
                    print(f"  🎯 Recall@100: {recall_change:+.3f} {'📈' if recall_change > 0 else '📉' if recall_change < 0 else '➡️'}")
                    print(f"  🎯 mAP: {map_change:+.3f} {'📈' if map_change > 0 else '📉' if map_change < 0 else '➡️'}")
                    print(f"  📊 候補数: {workload_change:+.0f} {'📈' if workload_change > 0 else '📉' if workload_change < 0 else '➡️'}")
            
            metrics_status.value = "<div style='color: green;'>✅ メトリクス表示完了</div>"
            
        except Exception as e:
            print(f"❌ メトリクス表示中にエラーが発生しました: {e}")
            metrics_status.value = "<div style='color: red;'>❌ エラー</div>"
            import traceback
            traceback.print_exc()

show_metrics_btn.on_click(show_metrics_trends)

# 表示
display(VBox([
    HBox([show_metrics_btn, metrics_status]),
    metrics_output
]))

## 9. 🎯 まとめと次のステップ

このノートブックを使って、以下のワークフローを実行できます：

### ✅ 実行手順
1. **パラメータ設定** → 解析条件を調整
2. **ハーモナイザー実行** → 地名収集・候補地抽出
3. **地図表示** → 結果を視覚的に確認
4. **AI診断実行** → 品質評価・改善提案
5. **レポート表示** → 詳細分析結果を確認
6. **改善計画作成** → 次回実行用の設定調整

### 🔄 継続改善サイクル
- 生成されたYAMLファイルに基づいてパラメータを調整
- メトリクスの履歴を確認して改善効果を測定
- 専門家の知見を反映した設定変更

### 📁 ファイル構成
- **入力**: `data/raw/` (OSM PBF, HydroRIVERS, GSW等)
- **中間結果**: `data/interim/acre_candidates.csv`
- **診断結果**: `data/output/inspector_reports/`
- **改善計画**: `actions/plan_*.yaml`

---

**🤖 このノートブックは非エンジニアでも安全に操作できるよう設計されています。**  
**❓ 質問やエラーが発生した場合は、開発チームにお気軽にお問い合わせください。**